# DCN-v2 Training Pipeline v2 (With Native Categorical Embeddings)

**Purpose**: Train Deep & Cross Network v2 with proper categorical feature handling.

**Key Improvements over v1**:
- Uses native categorical embeddings (not one-hot or target-encoded)
- Separate numerical and categorical feature pipelines
- Fixed logits/sigmoid handling

**Protocol**: Pair-programming approved (2026-02-16)

---

## Cell 1: Setup & Imports

In [ ]:
# Purpose: Mount Google Drive and set up environment
# Why: Data and model checkpoints stored on Drive for persistence
# Approved: 2026-02-16

from google.colab import drive
drive.mount('/content/drive')

# Set project path (adjust to your Drive folder)
PROJECT_PATH = '/content/drive/MyDrive/final_ex'  # <-- EDIT THIS

import sys
sys.path.insert(0, f'{PROJECT_PATH}/src')

In [ ]:
# Purpose: Import all required libraries
# Why: Standard ML/DL stack plus DCN-v2 modules
# Approved: 2026-02-16

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import os
import json

# DCN-v2 modules
from models.dcn_v2 import (
    DCNv2,
    compute_quantiles_for_binning,
    create_dcn_dataloaders,
    load_cardinalities,
    load_dcn_data,
)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Cell 2: Run Preprocessing (if needed)

Run this cell only if `dcn_X_*_num.parquet` files don't exist yet.

In [ ]:
# Purpose: Run DCN-v2 preprocessing if needed
# Why: Creates separate numerical and categorical feature files
# Approved: 2026-02-16

DATA_PATH = f'{PROJECT_PATH}/data/processed'

# Check if preprocessing is needed
if not os.path.exists(f'{DATA_PATH}/dcn_X_train_num.parquet'):
    print('Running DCN-v2 preprocessing...')
    from preprocessing.route_dcn_prep import prepare_dcn_data
    prepare_dcn_data()
    print('Preprocessing complete!')
else:
    print('Preprocessed files already exist.')

## Cell 3: Data Loading

In [ ]:
# Purpose: Create DataLoaders using new data utilities
# Why: Handles both numerical and categorical features properly
# Approved: 2026-02-16

BATCH_SIZE = 1024

train_loader, val_loader, test_loader, _ = create_dcn_dataloaders(
    data_path=DATA_PATH,
    batch_size=BATCH_SIZE,
    num_workers=2,
    include_reject=False,
)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

In [ ]:
# Purpose: Load feature info and cardinalities
# Why: Needed for model instantiation
# Approved: 2026-02-16

# Load cardinalities
cardinalities = load_cardinalities(DATA_PATH)
print(f'Categorical features: {len(cardinalities)}')
for feat, card in cardinalities.items():
    print(f'  {feat}: {card} categories')

# Load numerical feature names
X_train_num, _, _ = load_dcn_data(DATA_PATH, 'train')
NUMERICAL_FEATURES = list(X_train_num.columns)
print(f'\nNumerical features: {len(NUMERICAL_FEATURES)}')

## Cell 4: Quantile Computation

In [ ]:
# Purpose: Compute quantiles for SoftBinning initialization
# Why: Data-driven cutpoints for better convergence
# Approved: 2026-02-16

NUM_BINS = 10

# Compute quantiles for numerical features
quantiles_dict = {}
for feat in NUMERICAL_FEATURES:
    values = X_train_num[feat].values
    quantiles = compute_quantiles_for_binning(torch.tensor(values), NUM_BINS)
    quantiles_dict[feat] = quantiles

print(f'Computed quantiles for {len(quantiles_dict)} numerical features')

## Cell 5: Model Instantiation

In [ ]:
# Purpose: Instantiate DCN-v2 with categorical support
# Why: Native categorical embeddings instead of one-hot
# Approved: 2026-02-16

model = DCNv2(
    numerical_features=NUMERICAL_FEATURES,
    categorical_cardinalities=cardinalities,  # NOW POPULATED!
    num_bins=NUM_BINS,
    numerical_embed_dim=16,
    cross_layers=3,
    cross_rank=64,
    deep_hidden=[512, 256, 128],
    dropout=0.1,
)

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Embedding output dimension: {model.embedding.output_dim}')

In [ ]:
# Purpose: Initialize SoftBinning cutpoints with quantiles
# Why: Data-driven initialization for better convergence
# Approved: 2026-02-16

for feat in NUMERICAL_FEATURES:
    soft_bin = model.embedding.numerical_binning[feat]
    soft_bin.cutpoints.data = quantiles_dict[feat].to(device)

print('SoftBinning cutpoints initialized from training quantiles')

## Cell 6: Training Setup

In [ ]:
# Purpose: Set up loss function and optimizer
# Why: Class imbalance (~8% default rate) requires pos_weight
# Approved: 2026-02-16

# Load labels to compute class weight
y_train = pd.read_parquet(f'{DATA_PATH}/dcn_y_train.parquet')
n_positive = y_train.values.sum()
n_negative = len(y_train) - n_positive
pos_weight = torch.tensor([n_negative / n_positive], device=device)

print(f'Class distribution: {n_positive:.0f} positive ({100*n_positive/len(y_train):.1f}%)')
print(f'pos_weight: {pos_weight.item():.2f}')

# BCE with logits (numerically stable)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# AdamW optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

In [ ]:
# Purpose: Training and evaluation functions (FIXED version)
# Why: Properly handle both numerical and categorical features
# Approved: 2026-02-16

def train_epoch(model, loader, criterion, optimizer, device):
    """
    Train for one epoch.
    
    FIXED: Properly handles categorical features and extracts logits.
    """
    model.train()
    total_loss = 0.0
    n_batches = 0
    
    for num_vals, cat_vals, labels in loader:
        # Move to device
        num_vals = {k: v.to(device) for k, v in num_vals.items()}
        cat_vals = {k: v.to(device) for k, v in cat_vals.items()}
        labels = labels.to(device)
        
        # Forward pass - extract logits (before sigmoid)
        # NOTE: Model returns sigmoid, but we need logits for BCEWithLogitsLoss
        x0 = model.embedding(num_vals, cat_vals)
        cross_out = model.cross_network(x0)
        deep_out = model.deep_network(x0)
        combined = torch.cat([cross_out, deep_out], dim=1)
        logits = model.head(combined)
        
        # Compute loss
        loss = criterion(logits, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / n_batches


def evaluate(model, loader, device):
    """
    Evaluate model and compute AUC-ROC.
    
    FIXED: Properly handles categorical features.
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for num_vals, cat_vals, labels in loader:
            # Move to device
            num_vals = {k: v.to(device) for k, v in num_vals.items()}
            cat_vals = {k: v.to(device) for k, v in cat_vals.items()}
            
            # Forward pass (returns sigmoid probabilities)
            probs = model(num_vals, cat_vals)
            
            all_preds.extend(probs.cpu().numpy().flatten())
            all_labels.extend(labels.numpy().flatten())
    
    auc = roc_auc_score(all_labels, all_preds)
    return auc

## Cell 7: Training Loop

In [ ]:
# Purpose: Main training loop with early stopping
# Why: Prevent overfitting by monitoring validation AUC
# Approved: 2026-02-16

N_EPOCHS = 100
PATIENCE = 10
CHECKPOINT_PATH = f'{PROJECT_PATH}/outputs/models/dcn_v2_cat_best.pth'

# Create output directory
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

best_val_auc = 0.0
patience_counter = 0
history = {'train_loss': [], 'val_auc': []}

for epoch in range(N_EPOCHS):
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Evaluate
    val_auc = evaluate(model, val_loader, device)
    
    # Log
    history['train_loss'].append(train_loss)
    history['val_auc'].append(val_auc)
    
    print(f'Epoch {epoch+1:3d}/{N_EPOCHS} | Train Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f}')
    
    # Checkpointing
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auc': val_auc,
        }, CHECKPOINT_PATH)
        print(f'  --> New best! Saved checkpoint.')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)')
            break

print(f'\nBest validation AUC: {best_val_auc:.4f}')

## Cell 8: Final Evaluation

In [ ]:
# Purpose: Load best checkpoint and evaluate on test set
# Why: Final evaluation uses best model by validation AUC
# Approved: 2026-02-16

# Load best model
checkpoint = torch.load(CHECKPOINT_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded best model from epoch {checkpoint["epoch"]+1}')

# Evaluate on test set
test_auc = evaluate(model, test_loader, device)
print(f'\nTest AUC: {test_auc:.4f}')

# Compare to baselines
print('\n--- Comparison to Previous Routes ---')
print(f'Route A (VIME + CatBoost):     0.6912')
print(f'Route B (Raw + CatBoost):      0.7391')
print(f'Route F (Two-Model Ensemble):  0.7369')
print(f'DCN-v2 v1 (no categoricals):   0.7259')
print(f'DCN-v2 v2 (with categoricals): {test_auc:.4f}')

In [ ]:
# Purpose: Plot training history
# Why: Visual inspection of training dynamics
# Approved: 2026-02-16

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training loss
axes[0].plot(history['train_loss'])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss over Epochs')
axes[0].grid(True)

# Validation AUC
axes[1].plot(history['val_auc'])
axes[1].axhline(y=best_val_auc, color='r', linestyle='--', label=f'Best: {best_val_auc:.4f}')
axes[1].axhline(y=0.7391, color='g', linestyle='--', label='Route B: 0.7391')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation AUC')
axes[1].set_title('Validation AUC over Epochs')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/outputs/figures/dcn_v2_cat_training_history.png', dpi=150)
plt.show()

In [ ]:
# Purpose: Save training results
# Why: Reproducibility and comparison with other routes
# Approved: 2026-02-16

results = {
    'model': 'DCN-v2-categorical',
    'best_epoch': checkpoint['epoch'] + 1,
    'val_auc': best_val_auc,
    'test_auc': test_auc,
    'n_parameters': total_params,
    'n_numerical_features': len(NUMERICAL_FEATURES),
    'n_categorical_features': len(cardinalities),
    'embedding_dim': model.embedding.output_dim,
    'hyperparameters': {
        'num_bins': NUM_BINS,
        'numerical_embed_dim': 16,
        'cross_layers': 3,
        'cross_rank': 64,
        'deep_hidden': [512, 256, 128],
        'dropout': 0.1,
        'batch_size': BATCH_SIZE,
        'lr': 1e-3,
        'weight_decay': 1e-5,
        'patience': PATIENCE,
    },
    'cardinalities': cardinalities,
}

results_path = f'{PROJECT_PATH}/outputs/results/dcn_v2_cat_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved to {results_path}')

---

## Summary

### Changes from v1:
1. **Categorical embeddings**: 16 categorical features now have native PyTorch embeddings
2. **Separate pipelines**: Numerical (ClippedRobustScaler) and categorical (LabelEncoder)
3. **Larger embedding dimension**: ~1,959 dims vs 253 (more expressive)

### Next Steps:
1. **Phase 8.7**: Reject inference with updated model
2. **Phase 8.8**: Full evaluation and comparison

---

*DCN-v2 Training Pipeline v2 - With Native Categorical Embeddings - 2026-02-16*